Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
CC3084 - Data Science - sección 40

Integrantes:  
- Cristian Túnchez (231359)
- Dulce Ambrosio (231143)
- Daniel Chet (231177)
- Javier Linares (231135)

# 01. Carga y limpieza de los datos

**MAP - Charting Student Math Misunderstandings**

En este cuaderno se realiza el primer contacto con los datos de la competencia. El objetivo es responder tres preguntas antes de cualquier análisis: qué contiene el conjunto de datos, qué tan confiable es, y qué transformaciones necesita para poder ser analizado.

El trabajo se organiza de la siguiente manera:

1. **Carga y descripción de la estructura.** Cuántas observaciones y variables hay, y de qué tipo es cada una.
2. **Diagnóstico de calidad.** Valores faltantes, duplicados y consistencia interna.
3. **Limpieza y construcción de variables.** Las transformaciones necesarias y las variables nuevas que se derivan del texto.

El resultado se guarda en `data/processed/` y alimenta el cuaderno `02_eda.ipynb`.

## 1. Preparación del entorno

Se importan las bibliotecas necesarias y se añade la carpeta `src/` a la ruta de búsqueda de
Python para poder reutilizar las funciones del proyecto.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
sys.path.append(str(RAIZ / "src"))

from utils import DATA_RAW, guardar, set_estilo

set_estilo()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

print("Carpeta de datos crudos:", DATA_RAW)

Carpeta de datos crudos: /home/sebas/github/Data-Science/math-misconception-map/data/raw


## 2. Carga de los archivos

La competencia entrega tres archivos. El de entrenamiento es el único que trae las etiquetas y por tanto es el que se analiza. Los otros dos se cargan solo para entender el formato que se espera en la entrega.

In [2]:
train = pd.read_csv(DATA_RAW / "train.csv")
test = pd.read_csv(DATA_RAW / "test.csv")
sample_submission = pd.read_csv(DATA_RAW / "sample_submission.csv")

for nombre, df in [("train", train), ("test", test), ("sample_submission", sample_submission)]:
    print(f"{nombre:20s} filas: {df.shape[0]:>6,}   columnas: {df.shape[1]}")

train                filas: 36,696   columnas: 7
test                 filas:      3   columnas: 5
sample_submission    filas:      3   columnas: 2


In [3]:
print("Columnas de train:", list(train.columns))
print("Columnas de test: ", list(test.columns))
print()
print("Formato de entrega esperado:")
sample_submission

Columnas de train: ['row_id', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation', 'Category', 'Misconception']
Columnas de test:  ['row_id', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation']

Formato de entrega esperado:


,row_id,Category:Misconception
0,36696,True_Correct:NA False_Neither:NA False_Misconception:Incomplete
1,36697,True_Correct:NA False_Neither:NA False_Misconception:Incomplete
2,36698,True_Correct:NA False_Neither:NA False_Misconception:Incomplete


### ¿Qué se observa?

El conjunto de entrenamiento tiene **36,696 observaciones y 7 variables**. El archivo de prueba solo tiene 3 filas, porque es un marcador de posición (la competencia sustituye ese archivo por uno de aproximadamente 16,000 filas al momento de evaluar). Por eso todo el análisis se hace sobre `train`.

El archivo de prueba tiene 5 columnas en lugar de 7. Le faltan justamente `Category` y `Misconception`, que son las dos variables que hay que predecir.

El formato de entrega confirma que se espera una cadena `Category:Misconception` y se admiten **hasta tres predicciones separadas por espacio**. Es decir, la evaluación reconoce que la tarea es ambigua y premia tener la respuesta correcta entre las tres primeras candidatas.

Conviene notar que la documentación de la competencia no menciona la columna `row_id`, que sí está presente. Es simplemente el identificador de la fila.

## 3. Estructura de las variables

Se revisa el tipo de dato de cada variable, cuántos valores distintos toma y cuántos valores
faltantes tiene.

In [4]:
resumen = pd.DataFrame({
    "tipo_pandas": train.dtypes.astype(str),
    "valores_unicos": train.nunique(),
    "faltantes": train.isna().sum(),
    "pct_faltante": (train.isna().mean() * 100).round(2),
})
resumen

,tipo_pandas,valores_unicos,faltantes,pct_faltante
row_id,int64,36696,0,0.00
QuestionId,int64,15,0,0.00
QuestionText,object,15,0,0.00
MC_Answer,object,49,0,0.00
StudentExplanation,object,35726,0,0.00
Category,object,6,0,0.00
Misconception,object,35,26836,73.13


In [5]:
train.head(5)

,row_id,QuestionId,QuestionText,MC_Answer,StudentExplanation,Category,Misconception
0,0,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct,NaN
1,1,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third is not shaded.,True_Correct,NaN
2,2,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to understand.",True_Neither,NaN
3,3,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 goes into everything and 3 goes into nine,True_Neither,NaN
4,4,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 out of every 3 isn't coloured,True_Correct,NaN


### Tipo y nivel de medición de cada variable

El tipo que asigna pandas no basta para saber cómo tratar una variable. Por ejemplo `QuestionId` es un número entero, pero sumarlo o promediarlo no tiene sentido porque es una etiqueta. Por eso se clasifica cada variable por su nivel de medición.

| Variable | Tipo en pandas | Nivel de medición | Descripción |
| --- | --- | --- | --- |
| `row_id` | entero | Identificador | Número de fila, único en todo el archivo |
| `QuestionId` | entero | Identificador | Identifica la pregunta diagnóstica |
| `QuestionText` | texto | Texto libre | Enunciado de la pregunta, transcrito desde una imagen |
| `MC_Answer` | texto | Nominal | Opción de respuesta que el estudiante seleccionó |
| `StudentExplanation` | texto | Texto libre | Justificación escrita por el estudiante |
| `Category` | texto | Nominal | Relación entre la respuesta elegida y la explicación |
| `Misconception` | texto | Nominal | Error conceptual específico, solo cuando aplica |

Solo hay dos variables de texto libre y ninguna variable numérica de razón.

La variable `Misconception` tiene **73.13% de valores faltantes**. Antes de decidir qué hacer con ellos hay que averiguar si son un problema de calidad o una característica del diseño.